# APE

In [ ]:
import numpy as np
import pandas as pd
import random
import ollama
from tqdm import tqdm
import re
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

In [ ]:
ollama_model = "llama3:70b-instruct"

# Resampling (top5)

In [ ]:
# Generated instructions(Load Instruction Induction Prompt)
instruction_cleaned = pd.read_csv('inductions.csv')
instructions = list(instruction_cleaned['induction_set'])

In [ ]:
# Top Scored Instructions
top_instruction_index5 = [3, 17, 13, 11, 2]

top_instructions5 = []

for i in top_instruction_index5:
    worst = instruction_cleaned["induction_set"][i]
    top_instructions5.append(worst)

In [ ]:
# Prompt Format
resample_instructions = []

for i in range(len(top_instructions5)):
    resample_instruction = f"""Generate a variation of the following instruction while keeping the semantic meaning.
    input: {top_instructions5[i]}
    output: """
    resample_instructions.append(resample_instruction)

In [ ]:
resampled_instructions=[]
for resample_instruction in resample_instructions:
    
    response = ollama.chat(
        model=ollama_model,
        messages=[
        {"role": "system", "content": "You are a prompt Engineer. Generate *one* instruction without any other further explanations."},
        {'role': 'user', 'content': resample_instruction}],
        )
    print(response['message']['content'])
    resampled_instructions.append(response['message']['content'])

In [ ]:
resampled = pd.DataFrame({"ape_instructions":resampled_instructions})
resampled.to_csv('ape_instructions.csv')

# Second Evaluation

In [ ]:
# APE Instructions
ape = pd.read_csv('ape_instructions.csv')
ape_list = list(ape['ape_instructions'])

In [ ]:
# Load 2nd Evaluation Data
test = pd.read_excel('evaluation_2.xlsx', engine='openpyxl')
test_qq = list(test["Question"])
test_aa = list(test["Essay"])
test_output = list(test["Overall"])

In [ ]:
# Prompt Format
ape_instruction_dic = {}
number = 0

for instruction in ape_list:
    input_prompt = []
    
    for question, answer in zip(test_qq, test_aa):
        prompt = f"""{instruction}

Question: {question}
Answer Essay: {answer}
Score: """
        input_prompt.append(prompt)
    
    ape_instruction_dic[number] = input_prompt
    
    number = number + 1

In [ ]:
# Start Evaluation
result = {}

for i in tqdm(range(len(ape_list))):
    
    prompt_list = ape_instruction_dic[i]
    result_list = []
    
    for each_prompt in prompt_list:
        response = ollama.chat(
            model=ollama_model,
            messages=[
                {"role": "system", "content": "You are a helpful chatbot of this scoring task. Generate only an overall band score without further explanation. Generate the score as an integer."},
                {'role': 'user', 'content': each_prompt}],
            )
        print(response['message']['content'])
        result_list.append(response['message']['content'])
    
    result[i] = result_list

In [ ]:
result_df = pd.DataFrame.from_dict(result, orient='index')
result_df.to_excel('aperesult.xlsx')

# Scoring

In [ ]:
test_cc = pd.read_excel('evaluation_2.xlsx', engine='openpyxl')
test_ee = pd.read_excel('aperesult.xlsx', engine='openpyxl')

In [ ]:
test_cc_list = test_cc['Overall'].tolist()
test_ee_lists = test_ee.values.tolist()

In [ ]:
# QWK
qwk_list = []

from sklearn.metrics import cohen_kappa_score

for clean_test_e_list in test_ee_lists:
    kappa_quadratic = cohen_kappa_score(test_cc_list, clean_test_e_list, weights='quadratic', labels=[5,6,7,8,9])
    qwk_list.append(kappa_quadratic)
# IELTS labels=[5,6,7,8,9], ASAP++ labels=[0,1,2,3], ELLIPSE labels=[1,2,3,4,5]

In [ ]:
# Executive Accuracy
score_dict = {}
score_list = []

for i, clean_test_e_list in enumerate(test_ee_lists):
    diff = [pred == real for pred, real in zip(clean_test_e_list, test_cc_list)]
    score_list.append(sum(diff))
    score_dict[i] = diff

In [ ]:
# MSE
mse_scores = []

for clean_test_e_list in test_ee_lists:
    mse = mean_squared_error(test_cc_list, clean_test_e_list)
    mse_scores.append(mse)

In [ ]:
# rmse
rmse_scores = []

for clean_test_e_list in test_ee_lists:
    mse = mean_squared_error(test_cc_list, clean_test_e_list)
    rmse = np.sqrt(mse)
    rmse_scores.append(rmse)